# Resized model-input augmentation plot

One labelled class instance rendered as the model sees it: reference plus the three inverse-sort / polarity variants from `augmentation_inverse_sort_polarity.ipynb`. Each ERP image has its own colour scale and colour bar.

In [ ]:
import Pkg

ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

function find_repo_root(start_dir::AbstractString = pwd())
    candidates = unique(normpath.([
        start_dir,
        joinpath(start_dir, ".."),
        joinpath(start_dir, "..", ".."),
        joinpath(start_dir, "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from start_dir=$(start_dir).")
end

REPO_ROOT = find_repo_root()
MODEL_ENV_DIR = joinpath(REPO_ROOT, "notebooks", "model_test")
Pkg.activate(MODEL_ENV_DIR; io = devnull)

using CairoMakie
using DataFrames
using Printf: @sprintf

CairoMakie.activate!(type = "png")

function quiet_include(path::AbstractString)
    redirect_stdout(devnull) do
        redirect_stderr(devnull) do
            include(path)
        end
    end
end

if !isdefined(Main, :ERPImageProcessing)
    quiet_include(joinpath(REPO_ROOT, "scripts", "erp_image_processing.jl"))
end

if !isdefined(Main, :Week23ERPPatternExamples)
    quiet_include(joinpath(REPO_ROOT, "notebooks", "week_23", "erp_pattern_examples.jl"))
end

ImageProcessing = Main.ERPImageProcessing
PatternExamples = Main.Week23ERPPatternExamples
Week15 = PatternExamples.Week15

plot_target_trials = 200
plot_target_size = Week15.REAL_TARGET_SIZE
example_spec = (
    erp_class = "sigmoid",
    dataset_key = "fixations_dataset",
    channel_name = "ch096",
    sort_variable = "duration",
)

augmentation_variants = [
    (name = "reference", title = "model input\nnormal sort, normal polarity", inverse_sort = false, inverse_polarity = false),
    (name = "inverse_sort", title = "augmentation\ninverse sort", inverse_sort = true, inverse_polarity = false),
    (name = "inverse_polarity", title = "augmentation\ninverse polarity", inverse_sort = false, inverse_polarity = true),
    (name = "inverse_sort_inverse_polarity", title = "augmentation\ninverse sort + polarity", inverse_sort = true, inverse_polarity = true),
]

cellstr(x) = ismissing(x) || x === nothing ? "" : string(x)

function candidate_row_order(labels::DataFrame, spec)
    all_rows = collect(1:nrow(labels))
    exact = findall(all_rows) do i
        cellstr(labels.erp_class[i]) == spec.erp_class &&
        cellstr(labels.dataset_key[i]) == spec.dataset_key &&
        cellstr(labels.channel_name[i]) == spec.channel_name &&
        cellstr(labels.sort_variable[i]) == spec.sort_variable
    end
    same_class = findall(i -> cellstr(labels.erp_class[i]) == spec.erp_class, all_rows)
    return unique(vcat(exact, same_class, all_rows))
end

function fixed_model_chunk_indices(events::DataFrame, sort_col::Symbol; target_trials::Int)
    order = Week15.trial_sort_order(events, sort_col)
    n = length(order)
    full_chunk_count = div(n, target_trials)
    full_chunk_count >= 1 || error("Need at least $(target_trials) trials, got $(n).")
    remainder_count = rem(n, target_trials)

    full_bins = [Int[] for _ in 1:full_chunk_count]
    remainder = Int[]
    rank = 1
    while rank <= n
        progressed = false
        for bin in full_bins
            if length(bin) < target_trials && rank <= n
                push!(bin, order[rank])
                rank += 1
                progressed = true
            end
        end
        if remainder_count > 0 && length(remainder) < remainder_count && rank <= n
            push!(remainder, order[rank])
            rank += 1
            progressed = true
        end
        progressed || break
    end

    @assert length(full_bins[1]) == target_trials
    return full_bins[1], full_chunk_count, remainder_count
end

function sortvalues_from(events::DataFrame, sort_col::Symbol)
    values = events[!, sort_col]
    eltype(values) <: Number && return Float64.(values)
    return collect(values)
end

function pre_resize_augmented_image(data_time_trials::AbstractMatrix, events_trials::DataFrame, sort_col::Symbol;
        inverse_sort::Bool,
        inverse_polarity::Bool)

    @assert size(data_time_trials, 2) == nrow(events_trials)
    order = sortperm(sortvalues_from(events_trials, sort_col))
    inverse_sort && reverse!(order)

    data_ordered = Float32.(data_time_trials[:, order])
    inverse_polarity && (data_ordered .*= -1f0)
    data_z = ImageProcessing.zscore_timepoints(data_ordered)
    return Float32.(permutedims(data_z, (2, 1)))
end

function model_resized_image(data_time_trials::AbstractMatrix, events_trials::DataFrame, sort_col::Symbol, variant;
        target_size::Tuple{Int, Int})

    img = pre_resize_augmented_image(
        data_time_trials,
        events_trials,
        sort_col;
        inverse_sort = Bool(variant.inverse_sort),
        inverse_polarity = Bool(variant.inverse_polarity),
    )
    return Week15.process_erp_image(
        img,
        target_size;
        lowpass = true,
        sigma_factor = Week15.LOWPASS_SIGMA_FACTOR,
    )
end

function build_augmented_example(; spec = example_spec, target_trials::Int = plot_target_trials, target_size::Tuple{Int, Int} = plot_target_size)
    labels = PatternExamples.load_labelled_annotations()
    ctx = PatternExamples.build_reconstruction_context()

    for row_idx in candidate_row_order(labels, spec)
        row = labels[row_idx, :]
        origin = PatternExamples.origin_for_label(row, ctx)
        sort_col = Symbol(cellstr(row.sort_variable))
        sort_col in propertynames(origin.events) || continue
        nrow(origin.events) >= target_trials || continue

        trial_indices, full_chunk_count, remainder_count = fixed_model_chunk_indices(
            origin.events,
            sort_col;
            target_trials = target_trials,
        )
        events_part = origin.events[trial_indices, :]
        data_part = origin.data_time_trials[:, trial_indices]

        images = [(
            variant = variant,
            image = model_resized_image(data_part, events_part, sort_col, variant; target_size = target_size),
        ) for variant in augmentation_variants]

        return (
            erp_class = cellstr(row.erp_class),
            dataset_key = cellstr(row.dataset_key),
            dataset_label = cellstr(row.dataset_label),
            channel_name = cellstr(row.channel_name),
            sort_variable = String(sort_col),
            target_trials = target_trials,
            target_size = target_size,
            origin_n_trials = Int(origin.n_trials),
            full_chunk_count = full_chunk_count,
            remainder_count = remainder_count,
            images = images,
        )
    end

    error("No labelled example with at least $(target_trials) trials was found.")
end

function axis_ticks(n::Int)
    vals = unique([1, Int(round((n + 1) / 2)), n])
    return (vals, string.(vals))
end

function plot_augmented_model_inputs(sample)
    fig = Figure(size = (1560, 490), figure_padding = (18, 24, 14, 14))
    title = @sprintf(
        "%s | %s | ch=%s | sort=%s | %d trials -> %dx%d model input",
        replace(sample.erp_class, "_" => " "),
        PatternExamples.display_dataset_key(sample.dataset_key),
        sample.channel_name,
        sample.sort_variable,
        sample.target_trials,
        sample.target_size[1],
        sample.target_size[2],
    )
    Label(fig[0, 1:8], title;
        fontsize = 17,
        font = :bold,
        tellwidth = false,
        padding = (0, 0, 0, 8),
    )

    for (idx, item) in enumerate(sample.images)
        img = Float32.(item.image)
        clipped, colorrange, tick_vals, tick_labels, cmap = ImageProcessing.clipped_color_stats_quantile_zero_ticks(img)
        n_trials, n_time = size(clipped)
        col = 2idx - 1

        ax = Axis(fig[1, col];
            title = item.variant.title,
            titlesize = 14,
            xlabel = "time (resized)",
            ylabel = idx == 1 ? "sorted trials (resized)" : "",
            xticks = axis_ticks(n_time),
            yticks = axis_ticks(n_trials),
            xticklabelsize = 10,
            yticklabelsize = 10,
            xlabelsize = 12,
            ylabelsize = 12,
            aspect = DataAspect(),
        )

        hm = heatmap!(
            ax,
            1:n_time,
            1:n_trials,
            permutedims(Float32.(clipped), (2, 1));
            colormap = cmap,
            colorrange = colorrange,
            rasterize = true,
        )
        Colorbar(fig[1, col + 1], hm;
            ticks = (tick_vals, tick_labels),
            ticklabelsize = 9,
            width = 12,
        )
    end

    colgap!(fig.layout, 8)
    rowgap!(fig.layout, 8)
    resize_to_layout!(fig)
    return fig
end

sample = build_augmented_example()
plot_augmented_model_inputs(sample)